[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 07](README.md)

# Híbrido MPI + GPU

**Tema:** 07 · **Sesiones:** 32, 34 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo asignar procesos a dispositivos y solapar halos sin ocultar transferencias?


## Resultados de aprendizaje

- Construir un mapeo rank local–GPU.
- Descomponer halo, transferencia y comunicación.
- Distinguir MPI GPU-aware de staging por host.


## Modelo conceptual

El rank local, no el global, suele determinar la GPU dentro de un nodo.

Cuando hay más ranks que GPUs aparece compartición que debe ser intencional.

GPU-aware MPI puede aceptar buffers de dispositivo, pero soporte, ruta y sincronización deben verificarse.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "07"
NOTEBOOK = "07_hibrido/02_mpi_gpu.ipynb"
assert (ROOT / "curso" / "notebooks" / "07_hibrido" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Asignación de dispositivos

Se detecta sobresuscripción a partir de ranks locales y GPUs visibles.


In [ ]:
local_ranks, gpus = 6, 4
assignment = {rank: rank % gpus for rank in range(local_ranks)}
users = {gpu: [rank for rank, selected in assignment.items() if selected == gpu] for gpu in range(gpus)}
print(assignment)
for gpu, ranks in users.items(): print("GPU", gpu, "ranks", ranks, "compartida", len(ranks)>1)
assert set(assignment.values()) == set(range(gpus))


**Interpretación.** Si la política exige un rank por GPU, la asignación debe fallar en lugar de compartir silenciosamente.


## Volumen de halo

Se calcula comunicación por paso para una malla 3D descompuesta en una dimensión.


In [ ]:
ny, nz, layers, bytes_per_value = 512, 256, 2, 8
one_face = ny * nz * layers * bytes_per_value
interior_rank = 2 * one_face
print({"one_face_MiB": one_face/2**20, "interior_rank_MiB": interior_rank/2**20})
assert interior_rank == 2 * one_face


**Interpretación.** El modelo se combina con ancho de banda PCIe/NVLink y red para decidir staging, empaquetado y solapamiento.


## Práctica reproducible

1. Registrar rank global/local, bus id y modelo de GPU.
2. Comparar staging host y GPU-aware cuando ambos existan.
3. Medir interior, halo, red y sincronización con la misma entrada.


## Errores frecuentes

- Seleccionar GPU con rank global.
- Asumir GPU-aware por aceptar un puntero.
- Sincronizar todo el dispositivo y eliminar el solapamiento.

## Criterios de aceptación

- Mapeo proceso–GPU inequívoco.
- Halos validados contra referencia.
- Ruta de comunicación y sincronización documentadas.


## Referencias y material relacionado

- [Entornos de clúster](../../../topicos_avanzados/ENTORNOS_CLUSTER.md)
- [Protocolo hardware](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 07](README.md)
